In [ ]:
# ── Configuration ──
# Set artifacts_dir to a specific run's artifact path, or leave None
# to auto-discover the latest completed run from both demo-data and SAML-D.
artifacts_dir = None

In [ ]:
# Imports and run discovery
import os, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from tensorflow import keras
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import dash
from dash import dcc, html, dash_table, Input, Output, callback
import dash_bootstrap_components as dbc

# ── Discover completed runs from both artifact roots ──
BASE_PATH = Path(os.path.dirname(os.path.abspath('__file__')))
ARTIFACT_ROOTS = {
    'demo-data': BASE_PATH / 'artifacts' / 'runs',
    'saml-d': Path('/mnt/e/xx/saml-d/artifacts/runs'),
}

def find_valid_runs():
    """Scan both artifact roots for runs with required output files."""
    runs = []
    for source, root in ARTIFACT_ROOTS.items():
        if not root.exists():
            continue
        for run_dir in sorted(root.iterdir(), reverse=True):
            if not run_dir.is_dir():
                continue
            data_dir = run_dir / 'data'
            models_dir = run_dir / 'models'
            # Check required files exist
            has_edges = (data_dir / 'edges_td.csv').exists()
            has_embeddings = (data_dir / 'node_embeddings_fg.parquet').exists()
            has_model = any(models_dir.glob('gan_anomaly_*/anomaly_detector.keras')) if models_dir.exists() else False
            if has_edges and has_embeddings and has_model:
                try:
                    mtime = run_dir.stat().st_mtime
                except OSError:
                    mtime = 0
                runs.append({
                    'run_id': run_dir.name,
                    'source': source,
                    'path': str(run_dir),
                    'mtime': mtime,
                    'date': datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M'),
                })
    runs.sort(key=lambda r: r['mtime'], reverse=True)
    return runs

# ── Resolve artifacts_dir ──
SELECTED_RUN = None

if artifacts_dir:
    # Explicit path provided
    TRAINING_DATA_PATH = os.path.join(artifacts_dir, 'data')
    OUTPUT_PATH = os.path.join(artifacts_dir, 'data')
    MODELS_PATH = os.path.join(artifacts_dir, 'models')
    # Detect source from path
    _source = 'saml-d' if '/saml-d/' in str(artifacts_dir) else 'demo-data'
    _run_id = Path(artifacts_dir).name
    SELECTED_RUN = {'run_id': _run_id, 'source': _source, 'path': str(artifacts_dir)}
    print(f'Using provided artifacts_dir: {artifacts_dir}')
else:
    available_runs = find_valid_runs()
    if not available_runs:
        raise FileNotFoundError(
            'No completed runs found. Run the pipeline first, or set artifacts_dir manually.')
    # Auto-select latest run
    SELECTED_RUN = available_runs[0]
    artifacts_dir = SELECTED_RUN['path']
    TRAINING_DATA_PATH = os.path.join(artifacts_dir, 'data')
    OUTPUT_PATH = os.path.join(artifacts_dir, 'data')
    MODELS_PATH = os.path.join(artifacts_dir, 'models')

    print(f'Found {len(available_runs)} completed run(s):')
    for r in available_runs[:10]:
        tag = ' << selected' if r is SELECTED_RUN else ''
        print(f"  [{r['source']:>9}] {r['run_id'][:8]}…  ({r['date']}){tag}")

print(f'\nData source : {SELECTED_RUN["source"]}')
print(f'Run ID      : {SELECTED_RUN["run_id"]}')
print(f'Data path   : {TRAINING_DATA_PATH}')
print(f'Models path : {MODELS_PATH}')

In [3]:
# ── Data Loading & Enrichment ──

# Load core data
edges_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, 'edges_td.csv'))
nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, 'node_td.csv'))
node_embeddings = pd.read_parquet(os.path.join(OUTPUT_PATH, 'node_embeddings_fg.parquet'))

# Load model & threshold
model_dirs = sorted([d for d in os.listdir(MODELS_PATH) if d.startswith('gan_anomaly_')])
latest_model_dir = os.path.join(MODELS_PATH, model_dirs[-1])
model = keras.models.load_model(os.path.join(latest_model_dir, 'anomaly_detector.keras'))
threshold_val = float(np.load(os.path.join(latest_model_dir, 'threshold.npy')))

# Compute anomaly scores
emb_cols = [c for c in node_embeddings.columns if c.startswith('emb_')]
all_embeddings = node_embeddings[emb_cols].values
reconstructed = model.predict(all_embeddings, verbose=0)
anomaly_scores = np.mean(np.square(all_embeddings - reconstructed), axis=1)

node_embeddings['anomaly_score'] = anomaly_scores
node_embeddings['is_anomaly'] = anomaly_scores > threshold_val
score_min, score_max = anomaly_scores.min(), anomaly_scores.max()
node_embeddings['risk_score'] = (anomaly_scores - score_min) / (score_max - score_min)

# Map risk onto edges
node_risk_dict = node_embeddings.set_index('id')['risk_score'].to_dict()
node_anomaly_dict = node_embeddings.set_index('id')['is_anomaly'].to_dict()

edges_df['source_risk'] = edges_df['source'].map(node_risk_dict).fillna(0)
edges_df['target_risk'] = edges_df['target'].map(node_risk_dict).fillna(0)
edges_df['edge_risk'] = edges_df[['source_risk', 'target_risk']].max(axis=1)
edges_df['source_anomaly'] = edges_df['source'].map(node_anomaly_dict).fillna(False)
edges_df['target_anomaly'] = edges_df['target'].map(node_anomaly_dict).fillna(False)
edges_df['is_suspicious'] = edges_df['source_anomaly'] | edges_df['target_anomaly']

# Money flow per node
outgoing = edges_df.groupby('source').agg({'base_amt': 'sum', 'tran_id': 'count'}).rename(
    columns={'base_amt': 'outgoing_amt', 'tran_id': 'outgoing_count'})
incoming = edges_df.groupby('target').agg({'base_amt': 'sum', 'tran_id': 'count'}).rename(
    columns={'base_amt': 'incoming_amt', 'tran_id': 'incoming_count'})

node_money = node_embeddings[['id', 'anomaly_score', 'is_anomaly', 'risk_score']].copy()
if 'is_sar' in node_embeddings.columns:
    node_money['is_sar'] = node_embeddings['is_sar']
node_money = node_money.merge(outgoing, left_on='id', right_index=True, how='left')
node_money = node_money.merge(incoming, left_on='id', right_index=True, how='left')
node_money = node_money.fillna(0)
node_money['total_volume'] = node_money['outgoing_amt'] + node_money['incoming_amt']
node_money['total_transactions'] = node_money['outgoing_count'] + node_money['incoming_count']
node_money['net_flow'] = node_money['incoming_amt'] - node_money['outgoing_amt']

# Node type mapping
node_type_dict = nodes_df.set_index('id')['type'].to_dict()
node_money = node_money.merge(nodes_df[['id', 'type']], on='id', how='left')
edges_df['source_type'] = edges_df['source'].map(node_type_dict).fillna(-1).astype(int)
edges_df['target_type'] = edges_df['target'].map(node_type_dict).fillna(-1).astype(int)

# Loss/savings calculation
n_anomalies = int(node_embeddings['is_anomaly'].sum())
n_normal = len(node_embeddings) - n_anomalies
suspicious_txn_value = float(edges_df[edges_df['is_suspicious']]['base_amt'].sum())

DEMO_CONFIG = {
    'avg_loss_per_undetected_aml': 0.15,
    'investigation_cost_per_alert': 500,
    'false_positive_cost': 200,
    'regulatory_fine_multiplier': 3.0,
    'recovery_rate_detected': 0.70,
}

if 'is_sar' in node_embeddings.columns:
    tp = int(((node_embeddings['is_sar'] == 1) & (node_embeddings['is_anomaly'])).sum())
    fp = int(((node_embeddings['is_sar'] == 0) & (node_embeddings['is_anomaly'])).sum())
    fn = int(((node_embeddings['is_sar'] == 1) & (~node_embeddings['is_anomaly'])).sum())
    tn = int(((node_embeddings['is_sar'] == 0) & (~node_embeddings['is_anomaly'])).sum())
else:
    tp = int(n_anomalies * 0.10)
    fp = n_anomalies - tp
    fn = int(n_normal * 0.01)
    tn = n_normal - fn

cfg = DEMO_CONFIG
avg_suspicious_txn = suspicious_txn_value / max(n_anomalies, 1)
potential_loss_detected = tp * avg_suspicious_txn * cfg['avg_loss_per_undetected_aml']
recovered_amount = potential_loss_detected * cfg['recovery_rate_detected']
normal_txn_value = float(edges_df[~edges_df['is_suspicious']]['base_amt'].sum())
avg_normal_txn = normal_txn_value / max(n_normal, 1)
potential_loss_undetected = fn * avg_normal_txn * cfg['avg_loss_per_undetected_aml']
regulatory_fine_risk = potential_loss_undetected * cfg['regulatory_fine_multiplier']
investigation_cost = n_anomalies * cfg['investigation_cost_per_alert']
false_positive_cost = fp * cfg['false_positive_cost']
total_operational_cost = investigation_cost + false_positive_cost
net_savings = recovered_amount - total_operational_cost

precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * (precision * recall) / max(precision + recall, 1e-9)

print(f'Loaded {len(edges_df):,} transactions, {len(node_embeddings):,} nodes')
print(f'Anomalies: {n_anomalies:,}  Threshold: {threshold_val:.6f}')

I0000 00:00:1770131217.336314   41251 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9511 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-02-03 20:06:58.419662: I external/local_xla/xla/service/service.cc:163] XLA service 0x7ce0f8002100 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-03 20:06:58.419697: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9
2026-02-03 20:06:58.431081: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-03 20:06:58.467386: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801
I0000 00:00:1770131218.937485   41469 device_compiler.h:196] Compiled cluster using XLA!  This

Loaded 438,386 transactions, 7,347 nodes
Anomalies: 7,236  Threshold: 0.000157


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Build Dash App
# ══════════════════════════════════════════════════════════════════════════════

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.DARKLY])

# ── colour constants ──
CLR = dict(blue='#3498db', green='#2ecc71', red='#e74c3c', purple='#9b59b6',
           orange='#e67e22', yellow='#f1c40f', dark='#2c3e50', light='#ecf0f1')
RISK_COLORS = [CLR['green'], CLR['yellow'], CLR['orange'], CLR['red']]
RISK_LABELS = ['Low', 'Medium', 'High', 'Critical']

# ── helper: KPI card ──
def kpi_card(title, value, color, icon=''):
    return dbc.Card([
        dbc.CardBody([
            html.H6(title, className='text-muted mb-1', style={'fontSize': '0.85rem'}),
            html.H3(value, className='mb-0', style={'color': color, 'fontWeight': 'bold'}),
        ])
    ], className='shadow-sm', style={'borderLeft': f'4px solid {color}'})

# ══════════════════════════════════════════════════════════════════════════════
#  TAB 1 — Executive Summary
# ══════════════════════════════════════════════════════════════════════════════

# Risk distribution bar
risk_cat = pd.cut(node_embeddings['risk_score'], bins=[0, 0.25, 0.5, 0.75, 1.0],
                  labels=RISK_LABELS, include_lowest=True)
risk_counts = risk_cat.value_counts().reindex(RISK_LABELS).fillna(0)

fig_risk_bar = go.Figure(go.Bar(
    x=RISK_LABELS, y=risk_counts.values,
    marker_color=RISK_COLORS,
    text=[f'{int(v):,}' for v in risk_counts.values],
    textposition='outside'
))
fig_risk_bar.update_layout(template='plotly_dark', title='Risk Distribution',
                           yaxis_title='Nodes', margin=dict(t=40, b=30))

# Suspicious vs normal pie
susp_vol = float(edges_df[edges_df['is_suspicious']]['base_amt'].sum())
norm_vol = float(edges_df[~edges_df['is_suspicious']]['base_amt'].sum())
fig_vol_pie = go.Figure(go.Pie(
    labels=['Normal', 'Suspicious'],
    values=[norm_vol, susp_vol],
    marker_colors=[CLR['blue'], CLR['red']],
    hole=0.45,
    textinfo='label+percent',
    hovertemplate='%{label}<br>$%{value:,.0f}<extra></extra>'
))
fig_vol_pie.update_layout(template='plotly_dark', title='Volume: Normal vs Suspicious',
                          margin=dict(t=40, b=10))

# Top 10 risk nodes
top10 = node_money.nlargest(10, 'risk_score')
fig_top10 = go.Figure(go.Bar(
    y=[f"{r['id'][:10]}… ({r['risk_score']:.2f})" for _, r in top10.iterrows()],
    x=top10['total_volume'],
    orientation='h',
    marker_color=px.colors.sample_colorscale('Reds', top10['risk_score'].values),
    hovertemplate='Node: %{y}<br>Volume: $%{x:,.0f}<extra></extra>'
))
fig_top10.update_layout(template='plotly_dark', title='Top 10 Risk Nodes by Volume',
                        xaxis_title='Volume ($)', yaxis=dict(autorange='reversed'),
                        margin=dict(t=40, b=30, l=160))

# Anomaly score curve
sorted_scores = np.sort(anomaly_scores)
fig_score_curve = go.Figure()
fig_score_curve.add_trace(go.Scatter(
    x=list(range(len(sorted_scores))), y=sorted_scores,
    fill='tozeroy', fillcolor='rgba(52,152,219,0.2)',
    line=dict(color=CLR['blue'], width=2),
    hovertemplate='Node %{x}<br>Score: %{y:.6f}<extra></extra>'
))
fig_score_curve.add_hline(y=threshold_val, line_dash='dash', line_color='red',
                          annotation_text=f'Threshold {threshold_val:.6f}')
fig_score_curve.update_layout(template='plotly_dark', title='Anomaly Score Curve',
                              xaxis_title='Nodes (sorted)', yaxis_title='Score',
                              margin=dict(t=40, b=30))

tab1 = dbc.Container([
    dbc.Row([
        dbc.Col(kpi_card('Total Transactions', f'{len(edges_df):,}', CLR['blue']), md=3),
        dbc.Col(kpi_card('Total Volume', f'${edges_df["base_amt"].sum():,.0f}', CLR['green']), md=3),
        dbc.Col(kpi_card('Anomalies Detected', f'{n_anomalies:,}', CLR['red']), md=3),
        dbc.Col(kpi_card('Detection Rate', f'{100*node_embeddings["is_anomaly"].mean():.1f}%', CLR['purple']), md=3),
    ], className='mb-3 g-3'),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_risk_bar, config={'displayModeBar': True}), md=6),
        dbc.Col(dcc.Graph(figure=fig_vol_pie, config={'displayModeBar': True}), md=6),
    ], className='mb-3'),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_top10, config={'displayModeBar': True}), md=6),
        dbc.Col(dcc.Graph(figure=fig_score_curve, config={'displayModeBar': True}), md=6),
    ]),
], fluid=True)

# ══════════════════════════════════════════════════════════════════════════════
#  TAB 2 — Financial Impact
# ══════════════════════════════════════════════════════════════════════════════

# Confusion matrix heatmap
cm = np.array([[tn, fp], [fn, tp]])
fig_cm = px.imshow(cm, text_auto=True,
                   x=['Predicted Normal', 'Predicted Anomaly'],
                   y=['Actual Normal', 'Actual AML'],
                   color_continuous_scale='RdYlGn_r',
                   labels=dict(color='Count'))
fig_cm.update_layout(template='plotly_dark', title='Detection Matrix', margin=dict(t=40, b=30))

# Waterfall chart
fig_waterfall = go.Figure(go.Waterfall(
    x=['Suspicious<br>Value', 'Loss<br>Avoided', 'Recovered', 'Investigation<br>Cost',
       'FP Cost', 'Net Savings'],
    y=[suspicious_txn_value, -potential_loss_detected, recovered_amount,
       -investigation_cost, -false_positive_cost, net_savings],
    measure=['absolute', 'relative', 'relative', 'relative', 'relative', 'total'],
    connector_line_color='rgba(255,255,255,0.3)',
    increasing_marker_color=CLR['green'],
    decreasing_marker_color=CLR['red'],
    totals_marker_color=CLR['purple'],
    texttemplate='$%{y:,.0f}', textposition='outside'
))
fig_waterfall.update_layout(template='plotly_dark', title='Loss vs Savings Waterfall',
                            yaxis_title='Amount ($)', margin=dict(t=40, b=30))

# Gauge indicators
fig_gauges = make_subplots(rows=1, cols=3,
                           specs=[[{'type': 'indicator'}]*3],
                           subplot_titles=['Precision', 'Recall', 'F1 Score'])
for i, (name, val, color) in enumerate([
    ('Precision', precision, CLR['blue']),
    ('Recall', recall, CLR['green']),
    ('F1', f1, CLR['purple'])
], 1):
    fig_gauges.add_trace(go.Indicator(
        mode='gauge+number',
        value=val * 100,
        number_suffix='%',
        gauge=dict(axis=dict(range=[0, 100]),
                   bar_color=color,
                   steps=[dict(range=[0, 50], color='rgba(255,0,0,0.15)'),
                          dict(range=[50, 80], color='rgba(255,255,0,0.10)'),
                          dict(range=[80, 100], color='rgba(0,255,0,0.10)')]),
    ), row=1, col=i)
fig_gauges.update_layout(template='plotly_dark', height=280, margin=dict(t=40, b=10))

# Cost breakdown stacked bar
fig_cost = go.Figure()
fig_cost.add_trace(go.Bar(name='Investigation', x=['Operational Cost'],
                          y=[investigation_cost], marker_color=CLR['blue']))
fig_cost.add_trace(go.Bar(name='False-Positive', x=['Operational Cost'],
                          y=[false_positive_cost], marker_color=CLR['orange']))
fig_cost.update_layout(barmode='stack', template='plotly_dark',
                       title='Cost Breakdown', yaxis_title='$', margin=dict(t=40, b=30))

tab2 = dbc.Container([
    dbc.Row([
        dbc.Col(kpi_card('Recovered', f'${recovered_amount:,.0f}', CLR['green']), md=3),
        dbc.Col(kpi_card('Operational Cost', f'${total_operational_cost:,.0f}', CLR['orange']), md=3),
        dbc.Col(kpi_card('Net Savings', f'${net_savings:,.0f}',
                         CLR['green'] if net_savings >= 0 else CLR['red']), md=3),
        dbc.Col(kpi_card('Regulatory Risk', f'${regulatory_fine_risk:,.0f}', CLR['red']), md=3),
    ], className='mb-3 g-3'),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_cm), md=5),
        dbc.Col(dcc.Graph(figure=fig_waterfall), md=7),
    ], className='mb-3'),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_gauges), md=8),
        dbc.Col(dcc.Graph(figure=fig_cost), md=4),
    ]),
], fluid=True)

# ══════════════════════════════════════════════════════════════════════════════
#  TAB 3 — Network Graph
# ══════════════════════════════════════════════════════════════════════════════

import networkx as nx

# Pre-compute lookup dicts once for network hover texts
_vol_dict = node_money.set_index('id')['total_volume'].to_dict()
_txn_dict = node_money.set_index('id')['total_transactions'].to_dict()

def build_network_figure(filter_mode='all', max_nodes=300):
    """Build a plotly network graph."""
    edges_sorted = edges_df.sort_values(['edge_risk', 'base_amt'], ascending=[False, False])

    if filter_mode == 'suspicious':
        edges_subset = edges_sorted[edges_sorted['is_suspicious']].head(max_nodes * 3)
    elif filter_mode == 'high_risk':
        high_risk_nodes = set(node_embeddings[node_embeddings['risk_score'] > 0.75]['id'])
        edges_subset = edges_sorted[
            edges_sorted['source'].isin(high_risk_nodes) | edges_sorted['target'].isin(high_risk_nodes)
        ].head(max_nodes * 3)
    else:
        edges_subset = edges_sorted.head(max_nodes * 3)

    selected_nodes = set(edges_subset['source'].tolist() + edges_subset['target'].tolist())
    if len(selected_nodes) > max_nodes:
        top = node_embeddings[node_embeddings['id'].isin(selected_nodes)].nlargest(max_nodes, 'risk_score')['id']
        selected_nodes = set(top)
        edges_subset = edges_subset[
            edges_subset['source'].isin(selected_nodes) & edges_subset['target'].isin(selected_nodes)]

    G = nx.DiGraph()
    for _, r in edges_subset.iterrows():
        G.add_edge(r['source'], r['target'], amount=r['base_amt'], risk=r['edge_risk'])

    if G.number_of_nodes() == 0:
        fig = go.Figure()
        fig.add_annotation(text='No nodes match filter', xref='paper', yref='paper',
                           x=0.5, y=0.5, showarrow=False, font_size=20)
        fig.update_layout(template='plotly_dark')
        return fig

    pos = nx.spring_layout(G, k=3 / np.sqrt(G.number_of_nodes()), iterations=50, seed=42)

    # Edge traces
    edge_x, edge_y = [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines',
                            line=dict(width=0.5, color='rgba(150,150,150,0.3)'),
                            hoverinfo='none')

    # Node trace
    node_x = [pos[n][0] for n in G.nodes()]
    node_y = [pos[n][1] for n in G.nodes()]
    node_risk_vals = [node_risk_dict.get(n, 0) for n in G.nodes()]
    node_vol = [_vol_dict.get(n, 0) for n in G.nodes()]
    hover_texts = [
        f"ID: {n[:16]}<br>Risk: {node_risk_dict.get(n,0):.4f}<br>"
        f"Volume: ${_vol_dict.get(n,0):,.0f}<br>"
        f"Txns: {int(_txn_dict.get(n,0)):,}"
        for n in G.nodes()
    ]

    vol_max = max(node_vol) if node_vol else 1
    node_sizes = [6 + 20 * (v / vol_max) for v in node_vol]

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode='markers',
        marker=dict(size=node_sizes, color=node_risk_vals,
                    colorscale='RdYlGn_r', cmin=0, cmax=1,
                    colorbar=dict(title='Risk'), line=dict(width=0.5, color='white')),
        text=hover_texts, hoverinfo='text'
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        template='plotly_dark',
        title=f'Transaction Network ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)',
        showlegend=False, hovermode='closest',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        margin=dict(t=40, b=10, l=10, r=10), height=600
    )
    return fig

tab3 = dbc.Container([
    dbc.Row([
        dbc.Col([
            html.Label('Filter:', className='me-2'),
            dcc.Dropdown(id='network-filter',
                         options=[{'label': 'All Nodes', 'value': 'all'},
                                  {'label': 'Suspicious Only', 'value': 'suspicious'},
                                  {'label': 'High-Risk Neighborhood', 'value': 'high_risk'}],
                         value='all', clearable=False,
                         style={'color': '#000'}),
        ], md=4),
    ], className='mb-3'),
    dbc.Row([
        dbc.Col(dcc.Graph(id='network-graph', figure=build_network_figure('all')), md=12),
    ]),
], fluid=True)

# ══════════════════════════════════════════════════════════════════════════════
#  TAB 4 — Transaction Deep Dive
# ══════════════════════════════════════════════════════════════════════════════

# Amount histogram (log scale)
fig_amt_hist = px.histogram(edges_df, x='base_amt', nbins=80,
                            color_discrete_sequence=[CLR['blue']],
                            labels={'base_amt': 'Amount ($)'})
fig_amt_hist.update_layout(template='plotly_dark', title='Transaction Amount Distribution',
                           yaxis_type='log', yaxis_title='Count (log)',
                           margin=dict(t=40, b=30))

# Amount vs risk scatter (sample for performance)
scatter_sample = edges_df.sample(min(5000, len(edges_df)), random_state=42)
fig_amt_risk = px.scatter(scatter_sample, x='base_amt', y='edge_risk',
                          color='edge_risk', color_continuous_scale='RdYlGn_r',
                          hover_data=['source', 'target', 'base_amt', 'edge_risk'],
                          labels={'base_amt': 'Amount ($)', 'edge_risk': 'Risk'})
fig_amt_risk.update_layout(template='plotly_dark', title='Amount vs Risk',
                           margin=dict(t=40, b=30))

# Volume by tx_type
type_vol = edges_df.groupby('tx_type')['base_amt'].sum().sort_values(ascending=True).reset_index()
type_vol['tx_type'] = type_vol['tx_type'].astype(str)
fig_type_bar = px.bar(type_vol, y='tx_type', x='base_amt', orientation='h',
                      color='base_amt', color_continuous_scale='Blues',
                      labels={'base_amt': 'Volume ($)', 'tx_type': 'Tx Type'})
fig_type_bar.update_layout(template='plotly_dark', title='Volume by Transaction Type',
                           margin=dict(t=40, b=30))

# Risk heatmap by source-target type
heatmap_data = edges_df.pivot_table(values='edge_risk', index='source_type',
                                    columns='target_type', aggfunc='mean').fillna(0)
fig_heatmap = px.imshow(heatmap_data, color_continuous_scale='RdYlGn_r',
                        labels=dict(x='Target Type', y='Source Type', color='Avg Risk'),
                        x=[f'Type {c}' for c in heatmap_data.columns],
                        y=[f'Type {i}' for i in heatmap_data.index],
                        text_auto='.3f')
fig_heatmap.update_layout(template='plotly_dark', title='Avg Risk by Node-Type Pair',
                          margin=dict(t=40, b=30))

tab4 = dbc.Container([
    dbc.Row([
        dbc.Col([
            html.Label('Amount Range ($):', className='mb-1'),
            dcc.RangeSlider(
                id='amount-slider',
                min=0, max=float(edges_df['base_amt'].quantile(0.99)),
                step=100,
                value=[0, float(edges_df['base_amt'].quantile(0.99))],
                marks={0: '0', int(edges_df['base_amt'].quantile(0.99)): f'${int(edges_df["base_amt"].quantile(0.99)):,}'},
                tooltip={'placement': 'bottom', 'always_visible': True}
            ),
        ], md=12),
    ], className='mb-3'),
    dbc.Row([
        dbc.Col(dcc.Graph(id='amt-hist', figure=fig_amt_hist), md=6),
        dbc.Col(dcc.Graph(id='amt-risk-scatter', figure=fig_amt_risk), md=6),
    ], className='mb-3'),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_type_bar), md=6),
        dbc.Col(dcc.Graph(figure=fig_heatmap), md=6),
    ]),
], fluid=True)

# ══════════════════════════════════════════════════════════════════════════════
#  TAB 5 — Node Risk Profiles
# ══════════════════════════════════════════════════════════════════════════════

# Risk distribution histogram with percentile lines
fig_risk_hist = go.Figure()
fig_risk_hist.add_trace(go.Histogram(x=node_money['risk_score'], nbinsx=60,
                                     marker_color=CLR['blue'], opacity=0.75))
for p, clr in [(50, 'green'), (75, 'yellow'), (90, 'orange'), (95, 'red'), (99, 'darkred')]:
    val = float(np.percentile(node_money['risk_score'], p))
    fig_risk_hist.add_vline(x=val, line_dash='dash', line_color=clr,
                            annotation_text=f'P{p}: {val:.3f}')
fig_risk_hist.update_layout(template='plotly_dark', title='Risk Score Distribution',
                            xaxis_title='Risk Score', yaxis_title='Nodes',
                            margin=dict(t=40, b=30))

# Volume vs risk scatter
fig_vol_risk = px.scatter(
    node_money, x='total_volume', y='risk_score',
    color='is_anomaly', color_discrete_map={True: CLR['red'], False: CLR['blue']},
    hover_data=['id', 'total_volume', 'total_transactions', 'risk_score'],
    labels={'total_volume': 'Volume ($)', 'risk_score': 'Risk Score', 'is_anomaly': 'Anomaly'},
)
fig_vol_risk.update_layout(template='plotly_dark', title='Volume vs Risk',
                           margin=dict(t=40, b=30))

# Top 20 table data
top20 = node_money.nlargest(20, 'risk_score')[['id', 'risk_score', 'total_volume',
                                                'total_transactions', 'is_anomaly', 'net_flow']].copy()
top20['total_volume'] = top20['total_volume'].apply(lambda x: f'${x:,.0f}')
top20['risk_score'] = top20['risk_score'].round(4)
top20['net_flow'] = top20['net_flow'].apply(lambda x: f'${x:,.0f}')
top20['total_transactions'] = top20['total_transactions'].astype(int)

# Risk by node type box plot
fig_type_box = px.box(node_money, x='type', y='risk_score',
                      color='type', color_discrete_sequence=px.colors.qualitative.Set2,
                      labels={'type': 'Node Type', 'risk_score': 'Risk Score'})
fig_type_box.update_layout(template='plotly_dark', title='Risk by Node Type',
                           showlegend=False, margin=dict(t=40, b=30))

# Radar chart: high-risk vs low-risk averages
high_risk = node_money[node_money['risk_score'] > 0.75]
low_risk = node_money[node_money['risk_score'] <= 0.25]
radar_cats = ['Avg Volume', 'Avg Txns', 'Out Flow', 'In Flow', 'Net Flow Var']

def safe_norm(series, denom):
    return float(series.mean() / denom) if denom > 0 and len(series) > 0 else 0

vol_max = node_money['total_volume'].max() or 1
txn_max = node_money['total_transactions'].max() or 1
out_max = node_money['outgoing_amt'].max() or 1
in_max = node_money['incoming_amt'].max() or 1
nf_std = node_money['net_flow'].std() or 1

h_vals = [safe_norm(high_risk['total_volume'], vol_max),
          safe_norm(high_risk['total_transactions'], txn_max),
          safe_norm(high_risk['outgoing_amt'], out_max),
          safe_norm(high_risk['incoming_amt'], in_max),
          float(high_risk['net_flow'].std() / nf_std) if len(high_risk) > 1 else 0]
l_vals = [safe_norm(low_risk['total_volume'], vol_max),
          safe_norm(low_risk['total_transactions'], txn_max),
          safe_norm(low_risk['outgoing_amt'], out_max),
          safe_norm(low_risk['incoming_amt'], in_max),
          float(low_risk['net_flow'].std() / nf_std) if len(low_risk) > 1 else 0]

fig_radar = go.Figure()
fig_radar.add_trace(go.Scatterpolar(r=h_vals + [h_vals[0]], theta=radar_cats + [radar_cats[0]],
                                     fill='toself', name='High Risk', line_color=CLR['red'],
                                     fillcolor='rgba(231,76,60,0.2)'))
fig_radar.add_trace(go.Scatterpolar(r=l_vals + [l_vals[0]], theta=radar_cats + [radar_cats[0]],
                                     fill='toself', name='Low Risk', line_color=CLR['green'],
                                     fillcolor='rgba(46,204,113,0.2)'))
fig_radar.update_layout(template='plotly_dark', title='Risk Profile Comparison',
                        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
                        margin=dict(t=50, b=30))

# Lorenz curve
sorted_by_risk = node_money.sort_values('risk_score')
cum_vol = sorted_by_risk['total_volume'].cumsum() / sorted_by_risk['total_volume'].sum()
x_lorenz = np.linspace(0, 1, len(cum_vol))

fig_lorenz = go.Figure()
fig_lorenz.add_trace(go.Scatter(x=x_lorenz, y=cum_vol.values, fill='tonexty',
                                name='Actual', line_color=CLR['blue'],
                                fillcolor='rgba(52,152,219,0.2)'))
fig_lorenz.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line_dash='dash',
                                name='Equal', line_color='white'))
fig_lorenz.update_layout(template='plotly_dark', title='Risk Concentration (Lorenz)',
                         xaxis_title='Cumulative % Nodes', yaxis_title='Cumulative % Volume',
                         margin=dict(t=40, b=30))

tab5 = dbc.Container([
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_risk_hist), md=6),
        dbc.Col(dcc.Graph(figure=fig_vol_risk), md=6),
    ], className='mb-3'),
    dbc.Row([
        dbc.Col([
            html.H6('Top 20 Highest Risk Nodes', className='text-center mb-2'),
            dash_table.DataTable(
                data=top20.to_dict('records'),
                columns=[{'name': c, 'id': c} for c in top20.columns],
                sort_action='native', filter_action='native',
                page_size=10,
                style_table={'overflowX': 'auto'},
                style_header={'backgroundColor': '#2c3e50', 'color': 'white', 'fontWeight': 'bold'},
                style_cell={'backgroundColor': '#1a1a2e', 'color': 'white', 'fontSize': '12px',
                            'padding': '6px', 'textAlign': 'left'},
                style_data_conditional=[
                    {'if': {'filter_query': '{is_anomaly} = true'},
                     'backgroundColor': 'rgba(231,76,60,0.15)'},
                ],
            ),
        ], md=12),
    ], className='mb-3'),
    dbc.Row([
        dbc.Col(dcc.Graph(figure=fig_type_box), md=4),
        dbc.Col(dcc.Graph(figure=fig_radar), md=4),
        dbc.Col(dcc.Graph(figure=fig_lorenz), md=4),
    ]),
], fluid=True)

# ══════════════════════════════════════════════════════════════════════════════
#  App Layout — header shows loaded run info
# ══════════════════════════════════════════════════════════════════════════════

_src_badge_color = 'danger' if SELECTED_RUN['source'] == 'saml-d' else 'info'

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col([
            html.H2('AML Detection — Interactive Dashboard', className='text-center mt-3 mb-1'),
            html.Div([
                dbc.Badge(SELECTED_RUN['source'].upper(), color=_src_badge_color, className='me-2'),
                html.Span(f"Run {SELECTED_RUN['run_id'][:8]}…", className='text-muted me-3'),
                html.Span(f"{len(node_embeddings):,} nodes | {len(edges_df):,} txns | {n_anomalies:,} anomalies",
                          className='text-muted'),
            ], className='text-center mb-3'),
        ], md=12),
    ]),
    dbc.Tabs([
        dbc.Tab(tab1, label='Executive Summary', tab_id='tab-exec'),
        dbc.Tab(tab2, label='Financial Impact', tab_id='tab-fin'),
        dbc.Tab(tab3, label='Network Graph', tab_id='tab-net'),
        dbc.Tab(tab4, label='Transaction Deep Dive', tab_id='tab-txn'),
        dbc.Tab(tab5, label='Node Risk Profiles', tab_id='tab-node'),
    ], id='tabs', active_tab='tab-exec'),
], fluid=True)

# ── Callbacks ──

@app.callback(
    Output('network-graph', 'figure'),
    Input('network-filter', 'value')
)
def update_network(filter_mode):
    return build_network_figure(filter_mode)

@app.callback(
    [Output('amt-hist', 'figure'),
     Output('amt-risk-scatter', 'figure')],
    Input('amount-slider', 'value')
)
def update_txn_charts(amt_range):
    lo, hi = amt_range
    filtered = edges_df[(edges_df['base_amt'] >= lo) & (edges_df['base_amt'] <= hi)]

    hist = px.histogram(filtered, x='base_amt', nbins=80,
                        color_discrete_sequence=[CLR['blue']],
                        labels={'base_amt': 'Amount ($)'})
    hist.update_layout(template='plotly_dark', title=f'Amount Distribution ({len(filtered):,} txns)',
                       yaxis_type='log', yaxis_title='Count (log)', margin=dict(t=40, b=30))

    samp = filtered.sample(min(5000, len(filtered)), random_state=42) if len(filtered) > 0 else filtered
    scatter = px.scatter(samp, x='base_amt', y='edge_risk',
                         color='edge_risk', color_continuous_scale='RdYlGn_r',
                         hover_data=['source', 'target', 'base_amt', 'edge_risk'],
                         labels={'base_amt': 'Amount ($)', 'edge_risk': 'Risk'})
    scatter.update_layout(template='plotly_dark', title='Amount vs Risk (filtered)',
                          margin=dict(t=40, b=30))
    return hist, scatter

print('Dash app ready.')

In [5]:
# Launch the dashboard
# Runs inline in the notebook; also accessible at http://localhost:8050
app.run(jupyter_mode='inline', port=8050)